# 05 - Synthetic subscriber accounts (Salesforce Customer 360 analog)
Run once after `01_generate_synthetic_data`. Creates two raw tables in `classic_stable_l0m1dn_catalog.support_command_center`:
* `subscribers` - ~1M subscriber accounts (a CRM sample): tier, price, billing gateway, primary device, tenure, renewal date, auto-renew, watch hours last vs prior 30 days, segment.
* `ticket_accounts` - links every support ticket to the subscriber who raised it (`ticket_id -> subscriber_id`). Kept as a separate append-only table so the streaming source `support_tickets` is never modified.

Design so churn signals exist in the data:
* ticket tier/country always match the subscriber's tier/country
* skewed assignment -> some subscribers contact support repeatedly
* during incidents ~30% of tickets come from subscribers who already reported the same issue (repeat contacts)
* subscribers hit by incidents or with many contacts show a drop in watch hours

`04_live_event_generator` appends new `ticket_accounts` rows for newly generated tickets.

In [0]:
import numpy as np, pandas as pd
FQ = "classic_stable_l0m1dn_catalog.support_command_center"
rng = np.random.default_rng(2026)
AS_OF = pd.Timestamp("2026-10-27")

reg = spark.table(f"{FQ}.regions").toPandas()
tk = spark.sql(f"""
  SELECT t.ticket_id, t.created_at, t.country_code, t.subscriber_tier, t.device, s.issue_signature,
         a.anomaly_id IS NOT NULL AS incident_ticket
  FROM {FQ}.support_tickets t
  JOIN {FQ}_sdp.silver_tickets s USING (ticket_id)
  LEFT JOIN {FQ}_sdp.gold_ticket_anomalies a
    ON s.issue_signature = a.issue_signature AND t.created_at >= a.first_seen AND t.created_at < a.last_seen""").toPandas()
tk = tk.sort_values("created_at").reset_index(drop=True)
print(len(tk), "tickets;", int(tk.incident_ticket.sum()), "incident tickets")

TIERS = {"Free": (.30, 0.0), "Fan": (.35, 7.99), "Mega Fan": (.28, 11.99), "Ultimate Fan": (.07, 15.99)}
FIRST = ["Aiko", "Ben", "Carla", "Diego", "Emi", "Farah", "Gabriel", "Hana", "Ivan", "Julia", "Kenji", "Lucia", "Mateo", "Nina", "Omar", "Priya",
         "Rafael", "Sofia", "Takumi", "Valentina", "Wei", "Yuki", "Zara", "Lukas", "Chloe", "Arjun", "Isabela", "Hiro", "Lea", "Marco"]
LAST = ["Tanaka", "Silva", "Garcia", "Muller", "Martin", "Rossi", "Kim", "Santos", "Lopez", "Schmidt", "Dubois", "Sato", "Costa", "Nguyen",
        "Patel", "Fernandez", "Weber", "Suzuki", "Reyes", "Moreau", "Ito", "Oliveira", "Bianchi", "Cruz", "Smith", "Brown", "Wilson", "Rahman"]
DEVICES = ["Android TV", "Fire TV", "Roku", "Apple TV", "iOS", "Android", "Web", "PlayStation", "Xbox"]
DSHARE = np.array([.14, .08, .10, .04, .16, .22, .14, .08, .04])
EU_ADYEN = {"DE", "AT", "FR", "IT", "ES", "GB"}

rows = []
for r in reg.itertuples():
    n = int(round(r.subscribers / 16.5))   # ~1M accounts so tickets per subscriber are realistic
    tiers = rng.choice(list(TIERS), n, p=[v[0] for v in TIERS.values()])
    for t in tiers:
        rows.append((r.country_code, r.primary_language, t))
subs = pd.DataFrame(rows, columns=["country_code", "preferred_language", "tier"])
N = len(subs)
subs["subscriber_id"] = [f"SUB-{1000000 + i}" for i in range(N)]

# ---- link tickets to subscribers of the same country + tier (skewed -> repeat contacts)
subs["w"] = np.where(rng.random(N) < 0.10, rng.uniform(3, 6, N), 1.0)   # ~10% frequent contacters
groups = {k: (g.index.values, (g.w / g.w.sum()).values) for k, g in subs.groupby(["country_code", "tier"])}
sub_of = np.empty(len(tk), dtype=object)
prior_incident = {}  # (signature, country, tier) -> list of subscriber idx that already reported it
for i, r in enumerate(tk.itertuples()):
    idx, p = groups[(r.country_code, r.subscriber_tier)]
    key = (r.issue_signature, r.country_code, r.subscriber_tier)
    if r.incident_ticket and prior_incident.get(key) and rng.random() < 0.30:
        s = rng.choice(prior_incident[key])
    else:
        s = rng.choice(idx, p=p)
    if r.incident_ticket:
        prior_incident.setdefault(key, []).append(s)
    sub_of[i] = s
tk["sub_idx"] = sub_of.astype(int)
ticket_accounts = pd.DataFrame({"ticket_id": tk.ticket_id, "subscriber_id": subs.subscriber_id.values[tk.sub_idx.values], "linked_at": tk.created_at})

# ---- subscriber attributes
stats = tk.groupby("sub_idx").agg(n_tickets=("ticket_id", "size"), incident_tickets=("incident_ticket", "sum"),
                                  top_device=("device", lambda s: s.value_counts().index[0]))
subs = subs.join(stats, how="left")
subs[["n_tickets", "incident_tickets"]] = subs[["n_tickets", "incident_tickets"]].fillna(0)
subs["first_name"] = rng.choice(FIRST, N); subs["last_name"] = rng.choice(LAST, N)
subs["email"] = [f"{f.lower()}.{l.lower()}{i % 9973}@example.com" for f, l, i in zip(subs.first_name, subs.last_name, range(N))]
subs["monthly_price_usd"] = [TIERS[t][1] for t in subs.tier]
subs["primary_device"] = np.where(subs.top_device.notna(), subs.top_device, rng.choice(DEVICES, N, p=DSHARE))

def gateway(c, tier):
    if tier == "Free":
        return None
    opts = (["Stripe", "Adyen", "PayPal", "Apple IAP", "Google Play"], [.15, .3, .2, .15, .2]) if c in EU_ADYEN else \
           (["Stripe", "PayPal", "Apple IAP", "Google Play"], [.35, .2, .2, .25])
    return rng.choice(opts[0], p=opts[1])
subs["billing_gateway"] = [gateway(c, t) for c, t in zip(subs.country_code, subs.tier)]
tenure = np.clip(rng.gamma(1.6, 11, N).round().astype(int), 1, 96)
subs["tenure_months"] = tenure
subs["signup_date"] = [(AS_OF - pd.DateOffset(months=int(m))).date() - pd.Timedelta(days=int(d)) for m, d in zip(tenure, rng.integers(0, 28, N))]
renew_in = rng.integers(0, 31, N)
subs["renewal_date"] = np.where(subs.tier == "Free", None, [(AS_OF + pd.Timedelta(days=int(d))).date() for d in renew_in])
subs["auto_renew"] = np.where(subs.tier == "Free", False, rng.random(N) < 0.85)
subs["segment"] = np.select([tenure < 3, tenure > 24], ["New", "Loyal"], "Established")
subs["lifetime_value_usd"] = (subs.monthly_price_usd * subs.tenure_months).round(2)
base_hours = {"Free": 8, "Fan": 18, "Mega Fan": 30, "Ultimate Fan": 40}
prior = np.array([base_hours[t] for t in subs.tier]) * rng.lognormal(0, 0.45, N)
drop = np.zeros(N)
hit = (subs.incident_tickets.values > 0) & (rng.random(N) < 0.6)
drop[hit] = rng.uniform(0.3, 0.7, hit.sum())
many = (subs.n_tickets.values >= 3) & ~hit & (rng.random(N) < 0.4)
drop[many] = rng.uniform(0.1, 0.4, many.sum())
subs["watch_hours_prior_30d"] = prior.round(1)
subs["watch_hours_last_30d"] = (prior * rng.lognormal(0, 0.12, N) * (1 - drop)).round(1)
subs["marketing_opt_in"] = rng.random(N) < 0.55
subs["account_status"] = "Active"
subs["crm_source"] = "Salesforce (synthetic)"
subscribers = subs[["subscriber_id", "first_name", "last_name", "email", "country_code", "preferred_language", "tier", "monthly_price_usd",
                    "billing_gateway", "primary_device", "signup_date", "tenure_months", "renewal_date", "auto_renew", "segment",
                    "lifetime_value_usd", "watch_hours_prior_30d", "watch_hours_last_30d", "marketing_opt_in", "account_status", "crm_source"]]
print(f"{N:,} subscribers | {ticket_accounts.subscriber_id.nunique():,} raised tickets | "
      f"subscribers with 3+ tickets: {(subs.n_tickets >= 3).sum():,} | with incident tickets: {(subs.incident_tickets > 0).sum():,}")

In [0]:
from pyspark.sql import functions as F
sdf = spark.createDataFrame(subscribers.astype(object).where(pd.notnull(subscribers), None))
sdf = (sdf.withColumn("signup_date", F.col("signup_date").cast("date")).withColumn("renewal_date", F.col("renewal_date").cast("date"))
          .withColumn("monthly_price_usd", F.col("monthly_price_usd").cast("double")).withColumn("tenure_months", F.col("tenure_months").cast("int"))
          .withColumn("lifetime_value_usd", F.col("lifetime_value_usd").cast("double"))
          .withColumn("watch_hours_prior_30d", F.col("watch_hours_prior_30d").cast("double")).withColumn("watch_hours_last_30d", F.col("watch_hours_last_30d").cast("double"))
          .withColumn("auto_renew", F.col("auto_renew").cast("boolean")).withColumn("marketing_opt_in", F.col("marketing_opt_in").cast("boolean")))
sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.subscribers")
spark.createDataFrame(ticket_accounts).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.ticket_accounts")
spark.sql(f"COMMENT ON TABLE {FQ}.subscribers IS 'Synthetic subscriber accounts (Salesforce Customer 360 analog): tier, price, billing gateway, device, tenure, renewal, engagement.'")
spark.sql(f"COMMENT ON TABLE {FQ}.ticket_accounts IS 'Append-only link from support ticket to subscriber account.'")
print(spark.table(f"{FQ}.subscribers").count(), spark.table(f"{FQ}.ticket_accounts").count())